# Exercise 5 — export_portfolio

The `export_portfolio` function brings everything together: it writes `index.html` and one Markdown case study per project to a given directory. This produces a complete static portfolio site — upload `index.html` to GitHub Pages, Netlify, or Vercel and your portfolio is live.

In [ ]:
import pathlib, tempfile, os
from collections import Counter
from dataclasses import dataclass, field

@dataclass
class ProjectEntry:
    name: str; tagline: str; description: str; tech_stack: list
    github_url: str = ""; demo_url: str = ""; category: str = "AI Engineering"
    highlights: list = field(default_factory=list)

@dataclass
class PortfolioConfig:
    owner_name: str; title: str; bio: str; email: str; github_username: str
    linkedin_url: str = ""; projects: list = field(default_factory=list)

_P1 = ProjectEntry(
    name        = "AI Trading Bot",
    tagline     = "Paper-trading bot with sentiment + technical signals.",
    description = "Built over Days 89-96, this bot fetches OHLCV data, computes "
                  "technical indicators, scores news headlines with an LLM, applies "
                  "stop-loss and drawdown controls, and logs results daily.",
    tech_stack  = ["Python", "pandas", "Ollama", "SQLite"],
    github_url  = "https://github.com/testuser/ai-trading-bot",
    category    = "Finance",
    highlights  = ["Fully automated daily paper-trading loop",
                   "Kelly Criterion position sizing", "Stop-loss + drawdown gating"],
)
_P2 = ProjectEntry(
    name        = "Ops Agent",
    tagline     = "Autonomous multi-step ops agent with guardrails.",
    description = "Agent loop with tool routing, human-in-the-loop approval gates, "
                  "and task queue persistence.",
    tech_stack  = ["Python", "Ollama", "ChromaDB"],
    category    = "AI Agents",
    highlights  = ["Handles 5 operations autonomously", "Approval gate for destructive ops"],
)
_P3 = ProjectEntry(
    name        = "RAG Chatbot",
    tagline     = "Q&A chatbot grounded in your documents.",
    description = "Retrieval-augmented generation over a personal knowledge base.",
    tech_stack  = ["Python", "ChromaDB", "Ollama", "FastAPI"],
    github_url  = "https://github.com/testuser/rag-chatbot",
    demo_url    = "https://rag-chatbot.example.com",
    category    = "Text AI",
)

_CFG = PortfolioConfig(
    owner_name      = "Jane Doe",
    title           = "AI Engineer",
    bio             = "I build practical AI applications with Python. "
                      "100 days of AI engineering, shipped.",
    email           = "jane@example.com",
    github_username = "janedoe",
    linkedin_url    = "https://linkedin.com/in/janedoe",
    projects        = [_P1, _P2, _P3],
)
def _render_project_card(project):
    tech_tags = " ".join(f'<span class="tag">{t}</span>' for t in project.tech_stack)
    links = []
    if project.github_url: links.append(f'<a href="{project.github_url}">GitHub</a>')
    if project.demo_url:   links.append(f'<a href="{project.demo_url}">Demo</a>')
    links_html = " · ".join(links)
    card = (
        '      <div class="card">\n'
        f'        <h3>{project.name}</h3>\n'
        f'        <p class="category">{project.category}</p>\n'
        f'        <p>{project.tagline}</p>\n'
        f'        <div class="tags">{tech_tags}</div>\n'
    )
    if links_html:
        card += f'        <p class="links">{links_html}</p>\n'
    card += '      </div>'
    return card
def generate_portfolio_page(config):
    project_cards = "\n".join(_render_project_card(p) for p in config.projects)
    contact_parts = []
    if config.email:
        contact_parts.append(f'<a href="mailto:{config.email}">{config.email}</a>')
    if config.github_username:
        contact_parts.append(f'<a href="https://github.com/{config.github_username}">GitHub</a>')
    if config.linkedin_url:
        contact_parts.append(f'<a href="{config.linkedin_url}">LinkedIn</a>')
    contact_html = " · ".join(contact_parts)
    return (
        "<!DOCTYPE html>\n"
        '<html lang="en">\n'
        "<head>\n"
        '  <meta charset="UTF-8">\n'
        f"  <title>{config.owner_name} — {config.title}</title>\n"
        "  <style>\n"
        "    body { font-family: system-ui, sans-serif; margin: 0; }\n"
        "    header { background: #0f172a; color: white; padding: 60px 40px; }\n"
        "    h1 { font-size: 2.5rem; margin: 0 0 8px; }\n"
        "    .grid { display: grid; grid-template-columns: repeat(auto-fill, minmax(320px, 1fr)); gap: 24px; }\n"
        "    .card { border: 1px solid #e2e8f0; border-radius: 12px; padding: 24px; }\n"
        "    .tag { background: #f1f5f9; padding: 2px 8px; border-radius: 4px; font-size: 0.8rem; }\n"
        "  </style>\n"
        "</head>\n"
        "<body>\n"
        "  <header>\n"
        f"    <h1>{config.owner_name}</h1>\n"
        f'    <p class="subtitle">{config.title}</p>\n'
        f'    <p class="bio">{config.bio}</p>\n'
        f'    <p class="contact">{contact_html}</p>\n'
        "  </header>\n"
        "  <main>\n"
        "    <h2>Projects</h2>\n"
        '    <div class="grid">\n'
        f"{project_cards}\n"
        "    </div>\n"
        "  </main>\n"
        "</body>\n"
        "</html>"
    )
def generate_case_study(project):
    tech_list  = "\n".join(f"- {t}" for t in project.tech_stack)
    highlights = (
        "\n".join(f"- {h}" for h in project.highlights)
        if project.highlights else "- See project README for details"
    )
    links = []
    if project.github_url: links.append(f"- GitHub: {project.github_url}")
    if project.demo_url:   links.append(f"- Demo: {project.demo_url}")
    links_text = "\n".join(links) if links else "- See GitHub profile"
    return (
        f"# {project.name}\n\n"
        f"**Category:** {project.category}\n\n"
        f"## Overview\n\n"
        f"{project.tagline}\n\n"
        f"{project.description}\n\n"
        f"## Tech Stack\n\n"
        f"{tech_list}\n\n"
        f"## Key Achievements\n\n"
        f"{highlights}\n\n"
        f"## Links\n\n"
        f"{links_text}\n"
    )
def summarize_portfolio(config):
    categories = [p.category for p in config.projects]
    tech = []
    for p in config.projects: tech.extend(p.tech_stack)
    top_tech = [item for item, _ in Counter(tech).most_common(5)]
    return {
        "n_projects":         len(config.projects),
        "categories":         sorted(set(categories)),
        "n_categories":       len(set(categories)),
        "top_tech":           top_tech,
        "total_tech_entries": len(tech),
    }

def export_portfolio(config, output_dir):
    """Write portfolio files to output_dir and return list of written paths.

    Writes:
      output_dir/index.html
      output_dir/case_studies/{slug}.md   — one per project

    slug = project.name.lower().replace(' ', '_').replace('-', '_')

    Creates directories as needed. Returns list of absolute path strings,
    index.html first then case studies in project order.
    """
    out = pathlib.Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    written = []
    # TODO: write index.html, then case study .md files
    return written


### Checks

In [ ]:
checks = 0

with tempfile.TemporaryDirectory() as tmp:
    written = export_portfolio(_CFG, tmp)
    out = pathlib.Path(tmp)

    # 1 — returns a list of paths with correct count (1 + n_projects)
    try:
        expected_count = 1 + len(_CFG.projects)
        assert isinstance(written, list), f"expected list, got {type(written)}"
        assert len(written) == expected_count,             f"expected {expected_count} paths, got {len(written)}"
        checks += 1; print(f"✅ 1 export_portfolio returns {len(written)} paths (1 HTML + {len(_CFG.projects)} MD)")
    except Exception as e:
        print("❌ 1:", e)

    # 2 — index.html exists and contains owner_name
    try:
        index = out / "index.html"
        assert index.exists(), "index.html not created"
        content = index.read_text(encoding="utf-8")
        assert _CFG.owner_name in content, "owner_name not in index.html"
        checks += 1; print("✅ 2 index.html exists and contains owner_name")
    except Exception as e:
        print("❌ 2:", e)

    # 3 — case_studies/ directory exists with correct number of files
    try:
        cases = out / "case_studies"
        assert cases.is_dir(), "case_studies/ directory not created"
        md_files = list(cases.glob("*.md"))
        assert len(md_files) == len(_CFG.projects),             f"expected {len(_CFG.projects)} .md files, got {len(md_files)}"
        checks += 1; print(f"✅ 3 case_studies/ has {len(md_files)} .md files")
    except Exception as e:
        print("❌ 3:", e)

    # 4 — each case study starts with # project.name
    try:
        for p in _CFG.projects:
            slug = p.name.lower().replace(" ", "_").replace("-", "_")
            md_path = out / "case_studies" / f"{slug}.md"
            assert md_path.exists(), f"{slug}.md not found"
            content = md_path.read_text(encoding="utf-8")
            assert content.startswith(f"# {p.name}"),                 f"{slug}.md should start with '# {p.name}'"
        checks += 1; print("✅ 4 each case study starts with '# project.name'")
    except Exception as e:
        print("❌ 4:", e)

    # 5 — index.html is first in returned list; all paths exist
    try:
        assert "index.html" in written[0],             f"first path should be index.html, got {written[0]}"
        for path in written:
            assert pathlib.Path(path).exists(), f"path does not exist: {path}"
        checks += 1; print("✅ 5 index.html is first in list; all returned paths exist")
    except Exception as e:
        print("❌ 5:", e)

print(f"\n{checks}/5 checks passed!")
